# BETO Multi-Label Tag Classification - Training V2

Fine-tunes BETO (Spanish BERT) on synthetic + Groq story synopsis data.
**V2 improvements**: Focal Loss, deeper classifier head, targeted Groq data generation.

## Setup
1. Run all cells in order
2. The trained model will be saved to Google Drive
3. Upload `pytorch_model.bin` back to GitHub LFS or download locally

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install torch transformers scikit-learn numpy httpx

In [ ]:
!rm -rf -tagging-service
!git clone https://github.com/Xavi36772/-tagging-service.git
%cd /content/-tagging-service

In [ ]:
# Generate fresh diverse dataset (saves as train.json + val.json)
!python generate_dataset.py --samples 5000
print('Template dataset ready!')

In [ ]:
# Download existing Groq natural-language synopses
import urllib.request, json
url = "https://raw.githubusercontent.com/Xavi36772/-tagging-service/master/dataset/groq_natural.json"
urllib.request.urlretrieve(url, 'dataset/groq_natural.json')
with open('dataset/groq_natural.json', encoding='utf-8') as f:
    groq = json.load(f)
print(f'Downloaded {len(groq)} existing Groq entries')

## NEW: Generate Targeted Groq Data for Weak Tags

This step generates 1000-1500 additional synopses focused on the 21 weakest tags.
**Requires**: `GROQ_API_KEY` environment variable.
Takes ~10-15 minutes with rate limiting.

In [ ]:
import os
# Set your Groq API key here
os.environ['GROQ_API_KEY'] = 'your_groq_api_key_here'  # <-- REPLACE THIS

In [ ]:
# Generate targeted synopses for weak tags (~300 API calls, ~1500 synopses)
# Skip this cell if you don't have a Groq API key
!python generate_groq_targeted.py --calls 300 --batch-size 5 --delay 1.5

In [ ]:
# Merge ALL data sources: template + groq_natural + groq_targeted
import random, json
from collections import Counter
random.seed(42)

# Load all sources
template = []
for f in ['dataset/train.json', 'dataset/val.json']:
    try:
        with open(f, encoding='utf-8') as fp:
            template.extend(json.load(fp))
    except: pass

groq_natural = []
try:
    with open('dataset/groq_natural.json', encoding='utf-8') as fp:
        groq_natural = json.load(fp)
except: pass

groq_targeted = []
try:
    with open('dataset/groq_targeted.json', encoding='utf-8') as fp:
        groq_targeted = json.load(fp)
except: pass

print(f'Sources: template={len(template)}, groq_natural={len(groq_natural)}, groq_targeted={len(groq_targeted)}')

# Deduplicate by synopsis prefix
seen = set()
combined = []
for entry in template + groq_natural + groq_targeted:
    key = entry['synopsis'][:50]
    if key not in seen:
        seen.add(key)
        combined.append(entry)

random.shuffle(combined)
split = int(len(combined) * 0.8)
train = combined[:split]
val = combined[split:]

with open('dataset/train.json', 'w', encoding='utf-8') as f:
    json.dump(train, f, ensure_ascii=False, indent=2)
with open('dataset/val.json', 'w', encoding='utf-8') as f:
    json.dump(val, f, ensure_ascii=False, indent=2)

print(f'Combined: {len(combined)} | Train: {len(train)} | Val: {len(val)}')

# Show tag distribution
c = Counter()
for e in combined:
    for t in e['tags']:
        c[t] += 1
print(f'\nTag distribution (top 15):')
for tag, n in c.most_common(15):
    print(f'  {tag}: {n}')
print(f'\nWeakest 10:')
for tag, n in c.most_common()[-10:]:
    print(f'  {tag}: {n}')

In [ ]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    !nvidia-smi

In [ ]:
# Delete old model to train from scratch with new architecture
import os
for f in ['model/pytorch_model.bin', 'model/checkpoint.pt']:
    if os.path.exists(f):
        os.remove(f)
        print(f'Deleted {f}')
print('Ready for fresh training')

In [ ]:
# Train V2: Focal Loss + deeper head + dynamic weights + regularization
# ~45 mins on T4 GPU
!python train.py --epochs 20 --batch-size 32 --lr 2e-5 --data-dir dataset --dynamic-weights 2.0 --focal-loss --focal-gamma 2.0 --multisample-dropout --weight-decay 0.1 --label-smoothing 0.1

In [ ]:
# Evaluation
!python eval_model.py

In [ ]:
import shutil, os, json

# Save model to Google Drive
drive_dir = '/content/drive/MyDrive/beto_model_v2/'
os.makedirs(drive_dir, exist_ok=True)

for f in ['pytorch_model.bin', 'thresholds.npy', 'metrics.json', 'tokenizer.json', 'tokenizer_config.json']:
    src = f'model/{f}'
    if os.path.exists(src):
        shutil.copy(src, drive_dir)
        print(f'Copied {f} to Drive')

print(f'\nModel saved to: {drive_dir}')
print(f'\nFinal metrics:')
with open('model/metrics.json') as fp:
    m = json.load(fp)
print(f"  F1 Macro: {m['f1_macro']:.4f}")
print(f"  F1 Micro: {m['f1_micro']:.4f}")
print(f"  Hamming Loss: {m['hamming_loss']:.4f}")

## Post-Training Steps

1. **Download the model** from Google Drive: `beto_model_v2/pytorch_model.bin` (440MB)
2. **Place it in** `tagging-service/model/pytorch_model.bin`
3. **Update Railway**: Commit and push to trigger redeploy
   ```bash
   git add model/pytorch_model.bin
   git commit -m "feat: trained model v2 with Focal Loss + deeper head"
   git push origin master
   ```
4. **Update CACHE_BUST** in Dockerfile (increment the number)
5. **Verify**: `POST /predict-tags` should show improved accuracy